In [1]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np

def gurobi_solve(env_config):
    # Extract parameters from env_config
    T = env_config['periods']            # number of periods
    I0 = env_config['I0']                # initial inventories (dimension m-1)
    M = len(I0) + 1                      # number of stages
    alpha = env_config['alpha']
    p = env_config['p']                  # final product price at stage 0
    r = env_config['r']                  # unit cost array
    k = env_config['k']                  # backlog or lost sales cost
    h = env_config['h']                  # holding cost at each stage (m-1)
    c = env_config['c']                  # production capacities (m-1)
    L = env_config['L']                  # lead times (m-1)
    D = env_config['user_D']             # user-supplied demand for each period (length T)
    backlog = env_config.get('backlog', True)  # fallback to True if not specified

    # Construct full arrays for prices and costs
    p_full = np.append(p, r[:-1])  # Price array for all stages
    h_full = np.append(h, 0.0)     # Holding cost array for all stages (last stage = 0.0)

    Stages = range(M)
    Time = range(T)
    ProdStages = range(1, M)  # stages that produce

    model = gp.Model("multi_stage_inventory")

    # Decision variables
    R_vars = model.addVars(Time, range(M-1), name="R", lb=0)
    I_vars = model.addVars(Time, range(M-1), name="I", lb=0)
    Bvar = backlog
    if Bvar:
        B_vars = model.addVars(Time, Stages, name="B", lb=0)
    else:
        B_vars = None
    S_vars = model.addVars(Time, Stages, name="S", lb=0)

    # Initial inventory constraints (t=0)
    for s in range(M-1):
        # I[0,s] = I0[s] - S[0,s]
        model.addConstr(I_vars[0,s] == I0[s] - S_vars[0,s])

    # Inventory balance for t>0
    for t in Time:
        for s in range(M-1):
            if t > 0:
                inflow = 0
                if t - L[s] >= 0:
                    inflow = R_vars[t-L[s], s]
                # I[t,s] = I[t-1,s] + inflow - S[t,s]
                model.addConstr(I_vars[t,s] == I_vars[t-1,s] + inflow - S_vars[t,s])

    # Demand satisfaction at stage 0
    if Bvar:
        # With backlog:
        # For t=0: S[0,0] + B[0,0] = D[0]
        model.addConstr(S_vars[0,0] + B_vars[0,0] == D[0])
        # For t>0: S[t,0] + B[t,0] = D[t] + B[t-1,0]
        for t in range(1,T):
            model.addConstr(S_vars[t,0] + B_vars[t,0] == D[t] + B_vars[t-1,0])
    else:
        # Without backlog (lost sales):
        # S[t,0] ≤ D[t]
        for t in Time:
            model.addConstr(S_vars[t,0] <= D[t])

    # For stages > 0:
    # If backlog:
    # For t=0: S[0,s] + B[0,s] = R[0,s-1]
    # For t>0: S[t,s] + B[t,s] = R[t,s-1] + B[t-1,s]
    # If no backlog:
    # S[t,s] ≤ R[t,s-1]

    for t in Time:
        for s in range(1,M):
            if Bvar:
                if t == 0:
                    model.addConstr(S_vars[t,s] + B_vars[t,s] == R_vars[t,s-1])
                else:
                    model.addConstr(S_vars[t,s] + B_vars[t,s] == R_vars[t,s-1] + B_vars[t-1,s])
            else:
                model.addConstr(S_vars[t,s] <= R_vars[t,s-1])

    # Capacity constraints
    for t in Time:
        for s in ProdStages:
            if c[s-1] is not None:  # c is indexed from 0 for stage 1, so c[s-1]
                model.addConstr(S_vars[t,s] <= c[s-1])

    # Objective: maximize discounted profit
    discounted_profit_terms = []
    for t in Time:
        discount = alpha**t
        revenue = gp.quicksum(p_full[s]*S_vars[t,s] for s in Stages)
        order_cost = gp.quicksum(r[s]*R_vars[t,s] for s in range(M-1)) + r[M-1]*S_vars[t,M-1]

        backlog_cost = 0
        if Bvar:
            backlog_cost = gp.quicksum(k[s]*B_vars[t,s] for s in Stages)

        holding_cost = gp.quicksum(h_full[s]*I_vars[t,s] for s in range(M-1))

        period_profit = revenue - order_cost - backlog_cost - holding_cost
        discounted_profit_terms.append(discount * period_profit)

    model.setObjective(gp.quicksum(discounted_profit_terms), GRB.MAXIMIZE)

    # Solve the model
    model.optimize()

    # Print the optimal objective (reward)
    if model.status == GRB.OPTIMAL:
        print("Maximum Reward (Objective Value):", model.objVal)
    else:
        print("No optimal solution found.")
    return model.objVal


In [28]:
import random
import numpy as np
def generate_random_dist_params(dist):
    '''
    Generates random parameters for the customer demand distribution based on the dist value.
    '''
    dist_params = {}
    if dist == 1:  # Poisson distribution
        dist_params['mu'] = 15
    elif dist == 2:  # Binomial distribution mean 15 and variance 7.5
        dist_params['n'] = 30
        dist_params['p'] = 0.5
    elif dist == 3:  # Uniform random integer distribution mean 16, variance 100
        dist_params['low'] = 1
        dist_params['high'] = 31
    elif dist == 4:  # Geometric distribution
        dist_params['p'] = 1/15
    
    return dist_params
def generate_env_configs(num_configs=10):
    '''
    Generates a list of environment configurations with random parameters,
    where the length of lists is the same as the input lengths (hardcoded).
    
    Parameters:
    num_configs: int
        The number of different environment configurations to generate.
        
    Returns:
    List of dictionaries, each containing a set of parameters for the simulation.
    '''
    configs = []
    periods = 30 #fix the number of periods
    for _ in range(num_configs):
        dis=5
        config = {
            'periods': periods,  # Keep the number of periods the same as input
            'I0': [random.randint(150, 200) for _ in range(3)],  # Same length as input I0
            'p': 2,  # Random unit price between 1.5 and 3.0
            'r': [1.5, 1.0, 0.75, 0.5],  # Same length as input r
            'k': [0.10, 0.075, 0.05, 0.025],  # Same length as input k
            'h': [0.15, 0.10, 0.05],  # Same length as input h
            'c': [100, 90, 80],  # Same length as input c
            'L': [3, 5, 10],  # Same length as input L
            #'backlog': random.choice([True, False]),  # Randomly choose if unfulfilled orders are backlogged
            'dist': dis,  # Randomly choose distribution type(there are bugs in the code if we change this)
            'dist_param': generate_random_dist_params(dis),  # Generate random distribution parameters
            'alpha': round(random.uniform(0.9, 1.0), 2),  # Random discount factor between 0.9 and 1.0
            'seed_int': random.randint(0, 100),  # Random seed for the random state
            'user_D': np.random.poisson(lam=15, size=periods),  # User-supplied demand same length as periods
            '_max_rewards': 2000 
        }
        configs.append(config)
    
    return configs
 

In [23]:
new_inv= generate_env_configs(100)

In [24]:
new_inv[0]

{'periods': 30,
 'I0': [196, 197, 192],
 'p': 2,
 'r': [1.5, 1.0, 0.75, 0.5],
 'k': [0.1, 0.075, 0.05, 0.025],
 'h': [0.15, 0.1, 0.05],
 'c': [100, 90, 80],
 'L': [3, 5, 10],
 'dist': 3,
 'dist_param': {'low': 1, 'high': 31},
 'alpha': 0.98,
 'seed_int': 25,
 'user_D': array([21, 23, 15,  9, 24,  3, 25,  3, 23,  5, 35, 48, 31, 48, 44, 34, 30,
         1, 47, 13, 14, 18, 37, 41, 17, 47, 18,  2, 30, 16]),
 '_max_rewards': 2000}

In [4]:
"""This is an inv management problem. Finds best policy for the lowest cost.
On every iteration, improve priority_v1 over the priority_vX methods from previous iterations.
Make only small changes.
Try to make the code short.
"""
from scipy.optimize import minimize
import or_gym
import numpy as np
import funsearch
@funsearch.run
def evaluate(n,priority) -> int:
  results = solve(n,priority)
  max_value = sum(results)/len(results)
  print(max_value)
  return int(max_value)




def solve(n,priority ) :
  # Register environment
  def dfo_func(policy, env, *args):
    '''
    Runs an episode based on current base-stock model 
    settings. This allows us to use our environment for the 
    DFO optimizer.
    '''
    env.reset() # Ensure env is fresh
    rewards = []
    done = False
    while not done:
        action = priority(policy, env)
        state, reward, done, _ = env.step(action)
        rewards.append(reward)
        if done:
            break
            
    rewards = np.array(rewards)

    
    # Return negative of expected profit
    return -1 / env.num_periods * np.sum(rewards)
  
  def optimize_inventory_policy(env_name, fun,
    init_policy=None, env_config={}, method='Powell'):
    
    env = or_gym.make(env_name, env_config=env_config)
    
    if init_policy is None:
        init_policy = np.ones((env.num_stages-1)*2)
        
    # Optimize policy
    out = minimize(fun=fun, x0=init_policy, args=env, 
        method=method)
    
    policy = out.x.copy()
    
    # Policy must be positive integer
    policy = np.round(np.maximum(policy, 0), 0).astype(int)
    
    return policy, out
 
  
  policy_all=[]
  env_name='InvManagement-v1'
  for i in range(n):
    env_config=new_inv[i]
    policy, out = optimize_inventory_policy('InvManagement-v1',
    dfo_func,init_policy=priority(None,or_gym.make(env_name, env_config={})),env_config=env_config)
    policy_all.append(policy)
    
  policy=np.mean(policy_all,axis=0)
  print("Re-order levels: {}".format(policy))

  
  env_config =new_inv[n]
  env = or_gym.make(env_name, env_config=env_config)
  eps = 1000
  rewards = []
  for i in range(eps):
      env.reset()
      reward = 0
      while True:
          action = priority(policy, env)
          s, r, done, _ = env.step(action)
          reward += r
          if done:
              rewards.append(reward)
              break
  return rewards


array([12., 14., 20., 17., 25.,  8., 20., 14., 18., 16., 11., 24., 16.,
       15., 13., 16., 12., 13., 17., 15., 18., 13., 16., 19.,  7., 16.,
       14., 18., 13., 21.])

In [9]:
new_inv[0]

{'periods': 30,
 'I0': [169, 159, 177],
 'p': 2,
 'r': [1.5, 1.0, 0.75, 0.5],
 'k': [0.1, 0.075, 0.05, 0.025],
 'h': [0.15, 0.1, 0.05],
 'c': [100, 90, 80],
 'L': [3, 5, 10],
 'dist': 1,
 'dist_param': {'mu': 15},
 'alpha': 0.99,
 'seed_int': 10,
 'user_D': array([42, 25, 39, 44, 42, 42,  4, 15, 42, 38,  9,  0, 45, 35, 10, 15, 45,
        48, 27, 34,  2,  0, 49, 32, 33, 28, 24, 26, 18, 38]),
 '_max_rewards': 2000}

In [6]:

@funsearch.evolve
def priority(policy, env):   
    if policy is None:
        return np.ones(env.num_stages - 1) * 3  # Using 3 parameters per stage

    # Initialize state variables
    params = policy.reshape(-1, 3)  # [s, S, alpha] for each stage
    s, S, alpha = params.T

    # Calculate echelon inventory levels
    if env.period == 0:
        inv_ech = np.cumsum(env.I[env.period] + env.T[env.period])
    else:
        inv_ech = np.cumsum(env.I[env.period] + env.T[env.period] - env.B[env.period - 1, :-1])

    # Introduce dynamic demand adjustment
    demand_forecast = alpha * env.D[env.period] + (1 - alpha) * inv_ech

    # Unconstrained actions considering dynamic demand adjustment
    unc_actions = np.where(inv_ech < s, S - demand_forecast, 0)

    # Ensure actions respect constraints
    inv_const = np.hstack([env.I[env.period, 1:], np.Inf])
    actions = np.minimum(env.c, np.minimum(unc_actions, inv_const))
    
    return actions


In [12]:
@funsearch.evolve
def priority_old(policy, env):
  '''
  This is a re-order up-to policy for you to start. This means that for
  each node in the network, if the inventory at that node 
  falls below the level denoted by the policy, we will 
  re-order inventory to bring it to the policy level.
  
  Design new policy that fits the problem.and improve the score.
  '''
  if policy is None:
    return np.ones(3)
  # Get echelon inventory levels
  if env.period == 0:
    inv_ech = np.cumsum(env.I[env.period] +
      env.T[env.period])
  else:
    inv_ech = np.cumsum(env.I[env.period] +
      env.T[env.period] - env.B[env.period-1, :-1])
  # Get unconstrained actions
  unc_actions = policy - inv_ech
  unc_actions = np.where(unc_actions>0, unc_actions, 0)
  # Ensure that actions can be fulfilled by checking 
  # constraints
  inv_const = np.hstack([env.I[env.period, 1:], np.Inf])
  actions = np.minimum(env.c, np.minimum(unc_actions, inv_const))
  return actions



In [11]:

def priority_ss(policy, env):
  '''
  This is a re-order up-to policy for you to start. This means that for
  each node in the network, if the inventory at that node 
  falls below the level denoted by the policy, we will 
  re-order inventory to bring it to the policy level.
  
  Design new policy that fits the problem.and improve the score.
  '''
  #if policy is None, return the shape of the policy  
  #you should use this to first specify the shape of the policy in you implementation base on the env
  if policy is None :
    return np.ones((env.num_stages-1)*2)
  #split the policy into two parts to get s,S, first half is s, second half is S
  s = policy[:len(policy)//2]
  S = policy[len(policy)//2:]
  # Get echelon inventory levels
  if env.period == 0:
    inv_ech = np.cumsum(env.I[env.period] +
      env.T[env.period])
  else:
    inv_ech = np.cumsum(env.I[env.period] +
      env.T[env.period] - env.B[env.period-1, :-1])
  # Get unconstrained actions
  #for any inventory level below s, order up to S
  unc_actions = np.where(inv_ech < s, S-inv_ech, 0)
  # unc_actions = policy - inv_ech
  # unc_actions = np.where(unc_actions>0, unc_actions, 0)
  # Ensure that actions can be fulfilled by checking 
  # constraints
  inv_const = np.hstack([env.I[env.period, 1:], np.Inf])
  actions = np.minimum(env.c, np.minimum(unc_actions, inv_const))
  return actions



In [25]:
elva=evaluate(10,priority)

Re-order levels: [44.  58.5 22.9]
-78.15898761672506


In [26]:
elva=evaluate(10,priority_old)

Re-order levels: [10.4  3.   6.8]
-236.3049332384809


In [27]:
elva=evaluate(10,priority_ss)

Re-order levels: [15.9 29.8 27.4 42.7  8.1  8.6]
-165.7782966737888


In [22]:
 
  
np.mean([gurobi_solve(new_inv[i]) for i in range(10)])
 

Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (win64 - Windows 10.0 (19045.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-11800H @ 2.30GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 300 rows, 420 columns and 875 nonzeros
Model fingerprint: 0xfa1a116a
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e-02, 2e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [3e+00, 2e+02]
Presolve removed 112 rows and 23 columns
Presolve time: 0.00s
Presolved: 188 rows, 397 columns, 740 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.9550014e+04   4.047586e+03   0.000000e+00      0s
     321    6.9871157e+02   0.000000e+00   0.000000e+00      0s

Solved in 321 iterations and 0.01 seconds (0.00 work units)
Optimal objective  6.987115690e+02


Maximum Reward (Objective Value): 698.711569047288
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (win64 - Windows 10.0 (19045.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-11800H @ 2.30GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 300 rows, 420 columns and 875 nonzeros
Model fingerprint: 0x8c93e021
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e-02, 2e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 2e+02]
Presolve removed 111 rows and 22 columns
Presolve time: 0.00s
Presolved: 189 rows, 398 columns, 742 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.0768561e+04   4.233646e+03   0.000000e+00      0s
     333    7.8951650e+02   0.000000e+00   0.000000e+00      0s

Solved in 333 iterations and 0.01 seconds (0.00 work units)
Optimal objective  7.895165050e+02
Maximum Reward (Objective Value): 789.51

599.0236647473222